# Notebook 04: Advanced Prompt Workflows

## Learning objectives

By the end of this notebook, you will be able to:

1. Connect multiple model calls into a prompt chain.
2. Review and revise a draft against explicit criteria.
3. Compare alternatives using supplied evidence.
4. Verify model calculations using Python.
5. Identify where errors can propagate through a workflow.

**Keywords:** Chaining • Review • Alternatives • Verification • Evidence

## From a prompt to a workflow

A single prompt asks the model to perform a task.

A workflow connects several steps, with each step producing an output
that can be inspected, checked, or passed to the next step.

## 1. Setup

Start Ollama and confirm that your model is installed:

    ollama list

If needed, download the model:

     ollama pull qwen3:8b

 Run Jupyter on the same computer as Ollama.

 This notebook uses fictional workplace examples.

In [ ]:
%pip install requests pandas

In [1]:
import json
import time
import requests
import pandas as pd

from IPython.display import display, Markdown

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen3:8b"

# For this workflow demonstration, keep thinking disabled.
# Change this only if your selected model supports the setting.
THINK = False

TEMPERATURE = 0.2
MAX_GENERATED_TOKENS = 4096
REQUEST_TIMEOUT = 600

response = requests.get(
    f"{OLLAMA_URL}/api/tags",
    timeout=10
)
response.raise_for_status()

available_models = [
    item["name"]
    for item in response.json().get("models", [])
]

print("Installed models:", available_models)

if MODEL not in available_models:
    raise RuntimeError(
        f"{MODEL!r} is not installed. "
        "Select an installed model or download it using Ollama."
    )

Installed models: ['qwen3:8b', 'nomic-embed-text:latest', 'llama3.2:1b', 'qwen2.5:1.5b']


In [5]:
def ask_model(prompt, json_output=False):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "think": THINK,##########################################################################
        "stream": False,
        "options": {
            "temperature": TEMPERATURE,
            "num_predict": MAX_GENERATED_TOKENS
        }
    }

    if json_output:
        payload["format"] = "json"

    start = time.perf_counter()

    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/chat",
            json=payload,
            timeout=REQUEST_TIMEOUT
        )
        response.raise_for_status()

    except requests.RequestException as error:
        raise RuntimeError(
            f"Ollama request failed: {error}"
        ) from error

    data = response.json()

    result = {
        "answer": data.get("message", {}).get("content", ""),
        "seconds": round(time.perf_counter() - start, 2),
        "generated_tokens": data.get("eval_count"),
        "stop_reason": data.get("done_reason")
    }

    if not result["answer"].strip():
        raise RuntimeError("The model returned no final answer.")

    if result["stop_reason"] == "length":
        raise RuntimeError(
            "The response reached the generation limit. "
            "Increase MAX_GENERATED_TOKENS before continuing the workflow."
        )

    return result


def show_result(result):
    display(Markdown(result["answer"]))

    print(
        f"Time: {result['seconds']} seconds | "
        f"Generated tokens: {result['generated_tokens']}"
    )

## 2. Workflow A: Prompt chaining

A prompt chain executes separate model calls.

Our chain:

1. Extract requirements from source notes.
2. Identify gaps and dependencies.
3. Prepare a readiness summary.

Each stage receives the information needed for its task.

### Why inspect intermediate outputs?

An extraction error can affect every later stage.

Keep the original source available so that later outputs can be checked
against evidence rather than relying only on earlier model responses.

In [6]:
WORKSHOP_NOTES = """
The organization wants a two-hour introductory AI workshop.
There will be 40 participants with mixed Python experience.
The computer lab contains 25 computers.
Internet access may be unreliable.
Participants should complete two practical exercises.
The trainer has prepared the exercise instructions.
Software installation has not been confirmed.
The workshop date and technical support arrangements are not confirmed.
""".strip()

extract_prompt = f"""
Extract requirements and constraints from these notes.

Use only explicitly supplied facts.
Do not invent arrangements or turn suggestions into confirmed facts.
Use null where information is missing.

Return only JSON in this structure:
{{
  "duration_hours": 0,
  "participants": 0,
  "computers": 0,
  "practical_exercises": 0,
  "experience": "string",
  "internet_constraint": "string",
  "prepared_materials": "string",
  "software_ready": null,
  "workshop_date": null,
  "technical_support": null
}}

Source notes:
{WORKSHOP_NOTES}
""".strip()

extraction_result = ask_model(
    extract_prompt,
    json_output=True
)

requirements = json.loads(extraction_result["answer"])

if not isinstance(requirements, dict):
    raise ValueError("Expected a JSON object.")

print(json.dumps(requirements, indent=2))

{
  "duration_hours": 2,
  "participants": 40,
  "computers": 25,
  "practical_exercises": 2,
  "experience": "mixed Python experience",
  "internet_constraint": "Internet access may be unreliable",
  "prepared_materials": "The trainer has prepared the exercise instructions",
  "software_ready": null,
  "workshop_date": null,
  "technical_support": null
}


### Review checkpoint

Compare the extraction with the original notes.

Expected numerical facts:

- Duration: 2 hours.
- Participants: 40.
- Computers: 25.
- Practical exercises: 2.

Software readiness, workshop date, and technical support are unconfirmed.

The next cell checks numerical facts only.
You must still review the other fields for meaning and factual support.

In [19]:
expected_numbers = {
    "duration_hours": 2,
    "participants": 40,
    "computers": 25,
    "practical_exercises": 2
}

for field, expected in expected_numbers.items():
    actual = requirements.get(field)

    if type(actual) not in (int, float) or actual != expected:
        raise ValueError(
            f"Check extraction: {field} should be {expected}, "
            f"but the model returned {actual!r}."
        )

print("Numerical extraction checks passed.")

# Change to True after reviewing every field against the source notes.
EXTRACTION_REVIEWED = False

Numerical extraction checks passed.


In [20]:
gap_result = None

if not EXTRACTION_REVIEWED:
    print(
        "Review the extraction, then set "
        "EXTRACTION_REVIEWED = True in the previous cell."
    )

else:
    gap_prompt = f"""
Identify readiness gaps and dependencies for this workshop.

Original source:
{WORKSHOP_NOTES}

Reviewed extraction:
{json.dumps(requirements, indent=2)}

Return a table:
Gap or dependency | Supporting source fact | Recommended action

Requirements:
- Treat the original source as authoritative.
- Label actions as recommendations.
- Do not claim recommendations are already implemented.
- Do not invent owners, dates, or available resources.
""".strip()

    gap_result = ask_model(gap_prompt)
    show_result(gap_result)

Review the extraction, then set EXTRACTION_REVIEWED = True in the previous cell.


In [21]:
readiness_result = None

if gap_result is None:
    print("Complete the extraction review and gap-analysis step first.")

else:
    readiness_prompt = f"""
Prepare a workshop readiness summary for the training manager.

Original source:
{WORKSHOP_NOTES}

Gap analysis:
{gap_result["answer"]}

Requirements:
- Use the original source for factual claims.
- Treat the gap analysis as suggestions to evaluate.
- Keep the summary within 180 words.
- Separate confirmed arrangements from recommendations.
- Highlight decisions needed before the workshop.
- Do not invent dates or owners.

Output:
1. Confirmed arrangements
2. Readiness concerns
3. Decisions and next actions
""".strip()

    readiness_result = ask_model(readiness_prompt)
    show_result(readiness_result)

Complete the extraction review and gap-analysis step first.


## 3. Workflow B: Critique and revision

We will review and improve a deliberately flawed announcement.

Steps:

1. Identify unsupported claims.
2. Suggest source-supported corrections.
3. Produce a revised announcement.

A review generated by the same model is a second assessment.
It is not independent proof that the draft is correct.

In [22]:
FLAWED_DRAFT = """
Our AI workshop is fully ready.
All 40 participants will have individual computers.
Internet access is reliable and all software is installed.
Technical support has been assigned.
Participants will complete two practical exercises.
""".strip()

critique_prompt = f"""
Review this announcement against the source notes.

Source:
{WORKSHOP_NOTES}

Draft:
{FLAWED_DRAFT}

Check:
- Factual support.
- Whether unconfirmed arrangements are presented as confirmed.
- Whether resource availability is described accurately.

Return:
Draft claim | Problem | Source-supported correction

Do not add new facts.
""".strip()

critique_result = ask_model(critique_prompt)
show_result(critique_result)

Draft claim | Problem | Source-supported correction  
---|---|---  
"Our AI workshop is fully ready." | Overstates readiness; several key elements are unconfirmed. | "Our AI workshop is being prepared."  
"All 40 participants will have individual computers." | The lab has only 25 computers. | "There are 25 computers available for participants."  
"Internet access is reliable and all software is installed." | Internet access and software installation are unconfirmed. | "Internet access and software installation are pending confirmation."  
"Technical support has been assigned." | Technical support arrangements are unconfirmed. | "Technical support arrangements are pending confirmation."  
"Participants will complete two practical exercises." | This is supported. | "Participants will complete two practical exercises."

Time: 8.56 seconds | Generated tokens: 158


In [23]:
revision_prompt = f"""
Revise the announcement using the source notes and review.

Source:
{WORKSHOP_NOTES}

Original draft:
{FLAWED_DRAFT}

Review:
{critique_result["answer"]}

Requirements:
- Use only source-supported facts.
- Clearly identify unconfirmed arrangements.
- Do not promise individual computers.
- Use a neutral professional tone.
- Keep the announcement within 120 words.

Return only the revised announcement.
""".strip()

revision_result = ask_model(revision_prompt)
show_result(revision_result)

Our AI workshop is being prepared. There are 25 computers available for participants. Internet access and software installation are pending confirmation. Technical support arrangements are pending confirmation. Participants will complete two practical exercises. The trainer has prepared the exercise instructions. The workshop date has not been confirmed.

Time: 3.7 seconds | Generated tokens: 57


### Check the revision

A correct revision should:

- Remove the claim that everything is ready.
- State that there are 25 computers for 40 participants.
- Avoid promising reliable internet.
- Identify software installation and support as unconfirmed.
- Preserve the requirement for two practical exercises.

Evaluate the revision against the source, not just against the critique.

## 4. Workflow C: Compare alternatives

We will compare three fictional delivery options.

The instructor supplies the options and numerical values.
The model must not invent prices or ratings.

Selection rule:

1. Reject options that fail any mandatory requirement.
2. Choose the lowest-cost eligible option.
3. Explain remaining trade-offs.

Mandatory requirements:

- Cost must be at most INR 15,000.
- The option must support unreliable internet.
- At least 40 people must be able to participate.

"Capacity" means supported participants, not individual computers.

In [24]:
OPTIONS = [
    {
        "option": "A",
        "description": "Individual practice using cloud tools",
        "cost_inr": 12000,
        "supports_unreliable_internet": False,
        "capacity": 40
    },
    {
        "option": "B",
        "description": "Paired practice with prepared local materials",
        "cost_inr": 8000,
        "supports_unreliable_internet": True,
        "capacity": 40
    },
    {
        "option": "C",
        "description": "Additional rented computers with local materials",
        "cost_inr": 20000,
        "supports_unreliable_internet": True,
        "capacity": 40
    }
]

display(pd.DataFrame(OPTIONS))

comparison_prompt = f"""
Select a workshop delivery option using the supplied data.

Options:
{json.dumps(OPTIONS, indent=2)}

Mandatory requirements:
- Cost must be at most INR 15,000.
- Must support unreliable internet.
- Capacity must be at least 40 participants.

Selection rule:
Reject options failing any requirement.
Choose the lowest-cost eligible option.

Return:
1. An eligibility table with reasons.
2. The selected option.
3. A brief explanation of trade-offs.

Use only supplied data.
Do not invent additional costs or claim that an option guarantees learning.
""".strip()

comparison_result = ask_model(comparison_prompt)
show_result(comparison_result)

,option,description,cost_inr,supports_unreliable_internet,capacity
0,A,Individual practice using cloud tools,12000,False,40
1,B,Paired practice with prepared local materials,8000,True,40
2,C,Additional rented computers with local materials,20000,True,40


### 1. Eligibility Table with Reasons

| Option | Eligible? | Reason |
|--------|----------|--------|
| A      | **No**   | Does **not** support unreliable internet |
| B      | **Yes**  | Cost is within limit (≤15,000), supports unreliable internet, and capacity is ≥40 |
| C      | **No**   | Cost exceeds 15,000 (20,000 > 15,000) |

---

### 2. Selected Option

**Option B**

---

### 3. Brief Explanation of Trade-offs

Option B is the lowest-cost option that meets all the requirements. It supports unreliable internet and has a capacity of 40 participants. While it is less expensive than Option C, it does not include additional rented computers, which may be a consideration if more resources are needed. Option A is excluded because it does not support unreliable internet, which is a mandatory requirement.

Time: 7.04 seconds | Generated tokens: 208


## 5. Verify the selection using Python

The model can explain the decision.

Python can independently apply the numerical eligibility rules.

This check is independent of the model's explanation because it uses
the original option data directly.

In [14]:
eligible_options = [
    option
    for option in OPTIONS
    if option["cost_inr"] <= 15000
    and option["supports_unreliable_internet"] is True
    and option["capacity"] >= 40
]

if not eligible_options:
    selected_option = None
    print("No option satisfies all mandatory requirements.")

else:
    selected_option = min(
        eligible_options,
        key=lambda option: option["cost_inr"]
    )

    print("Eligible options:", [
        option["option"]
        for option in eligible_options
    ])

    print("Expected selection:", selected_option["option"])
    print("Cost:", selected_option["cost_inr"], "INR")

print("\nCompare this result with the model's recommendation.")

Eligible options: ['B']
Expected selection: B
Cost: 8000 INR

Compare this result with the model's recommendation.


## 6. Workflow D: Verify a calculation

A manager asks for the minimum duration of a project with parallel tasks.

We will:

1. Ask the model for an answer.
2. Calculate the duration independently in Python.
3. Check whether the model's final values match.

This verifies the answer for this specific task.
It does not establish general model reliability.

In [25]:
schedule_prompt = """
Calculate the minimum project duration.

Tasks:
- A takes 2 days.
- B takes 4 days and starts after A finishes.
- C takes 3 days and starts after A finishes.
- B and C can run in parallel.
- D takes 2 days and starts after both B and C finish.

Assume adequate resources, no gaps, and zero transition time.
The deadline is 7 days.

Return only JSON:
{
  "duration_days": 0,
  "deadline_met": false,
  "calculation": "brief calculation"
}
""".strip()

schedule_result = ask_model(
    schedule_prompt,
    json_output=True
)

schedule_answer = json.loads(schedule_result["answer"])

print(json.dumps(schedule_answer, indent=2))

{
  "duration_days": 7,
  "deadline_met": true,
  "calculation": "A (2 days) starts first. B and C start after A, and run in parallel. B (4 days) and C (3 days) finish on day 6. D starts after both B and C, taking 2 days, finishing on day 7. Deadline is met."
}


In [26]:
# Independently compute the answer from the original task durations.
# Lets assume this is the TASK Duration
A = 2
B = 4
C = 3
D = 2
DEADLINE = 7

expected_duration = A + max(B, C) + D
expected_deadline_met = expected_duration <= DEADLINE

schema_valid = (
    isinstance(schedule_answer, dict)
    and type(schedule_answer.get("duration_days")) in (int, float)
    and type(schedule_answer.get("deadline_met")) is bool
    and isinstance(schedule_answer.get("calculation"), str)
)

answer_correct = (
    schema_valid
    and schedule_answer["duration_days"] == expected_duration
    and schedule_answer["deadline_met"] is expected_deadline_met
)

print("Expected duration:", expected_duration, "days")
print("Expected deadline decision:", expected_deadline_met)
print("Model final values correct:", answer_correct)

print("\nReview the model's calculation separately.")

Expected duration: 8 days
Expected deadline decision: False
Model final values correct: False

Review the model's calculation separately.


## 7. Understand error propagation

| Workflow stage | Possible failure | Useful check |
|---|---|---|
| Extraction | A fact is omitted or changed | Compare with the source |
| Gap analysis | A suggestion becomes a supposed fact | Require evidence and labels |
| Drafting | Unsupported details are introduced | Review factual claims |
| Critique | The reviewer misses an error | Use a specific rubric |
| Selection | A mandatory constraint is ignored | Apply rules in Python |
| Final response | A correct number has an incorrect explanation | Check both independently |

### Practical rule

Do not pass an unreviewed intermediate answer through a long chain
and assume that later stages will repair it.

Add checkpoints where mistakes would materially affect the outcome.

## 8. Participant challenge

Build a two-stage workflow for this fictional scenario.

Facts:

- A customer demonstration is planned for 20 September 2026.
- The interface is complete.
- Integration testing is pending.
- One API dependency is unavailable.
- Nisha will contact the API provider by 17 September 2026.
- The dependency recovery date is unknown.

Stage 1:
Identify confirmed progress, risks, and missing information.

Stage 2:
Use the source facts and Stage 1 output to prepare a manager update.

Requirements:

- Keep facts separate from recommendations.
- Do not invent a recovery date.
- Do not guarantee that the demonstration will proceed.
- Keep the final update within 150 words.

Inspect Stage 1 before running Stage 2.

In [17]:
DEMO_FACTS = """
A customer demonstration is planned for 20 September 2026.
The interface is complete.
Integration testing is pending.
One API dependency is unavailable.
Nisha will contact the API provider by 17 September 2026.
The dependency recovery date is unknown.
""".strip()

stage_1_prompt = """
REPLACE THIS with your Stage 1 instructions.
""".strip()

participant_stage_1 = None

if "REPLACE THIS" in stage_1_prompt:
    print("Write your Stage 1 instructions first.")

else:
    participant_stage_1 = ask_model(
        f"{stage_1_prompt}\n\nSource facts:\n{DEMO_FACTS}"
    )

    show_result(participant_stage_1)

Write your Stage 1 instructions first.


In [18]:
stage_2_instructions = """
REPLACE THIS with your Stage 2 instructions.
""".strip()

# Set True after reviewing Stage 1 against the source facts.
STAGE_1_REVIEWED = False

if participant_stage_1 is None:
    print("Complete Stage 1 first.")

elif not STAGE_1_REVIEWED:
    print("Review Stage 1, then set STAGE_1_REVIEWED = True.")

elif "REPLACE THIS" in stage_2_instructions:
    print("Write your Stage 2 instructions first.")

else:
    stage_2_prompt = f"""
{stage_2_instructions}

Original source facts:
{DEMO_FACTS}

Reviewed Stage 1 output:
{participant_stage_1["answer"]}

Treat the original source facts as authoritative.
""".strip()

    participant_stage_2 = ask_model(stage_2_prompt)
    show_result(participant_stage_2)

Complete Stage 1 first.


## 9. Choose a workflow for your work

| Your task | Useful workflow |
|---|---|
| Convert notes into a plan | Extract → Review → Plan |
| Improve a report | Draft → Critique → Revise |
| Select among alternatives | Compare → Check constraints → Recommend |
| Produce a numerical answer | Calculate → Verify with Python → Explain |
| Prepare a grounded update | Extract facts → Identify gaps → Draft |

## Reflection

1. Which intermediate output needs review?
2. What should remain available as the original evidence?
3. Which checks can Python perform?
4. Which checks require human judgment?
5. Does the workflow improve enough to justify extra model calls?

## Key takeaways

- Give each stage a clear purpose.
- Preserve the original evidence.
- Inspect intermediate outputs.
- Separate suggestions from confirmed facts.
- Verify deterministic rules and calculations with code.
- Use additional calls when they improve the outcome.

## Next notebook

Notebook 05: Workplace Prompt Templates

We will create reusable templates for writing, meetings, planning,
analysis, coding, research, and teaching.